# Data Preparation: Standardizing Column Names
Here we load the raw CSV files, standardize their column names (lowercase, remove special characters, replace spaces with underscores), and save the cleaned versions to `data/02_processed/`.

In [51]:
import pandas as pd
import numpy as np

In [52]:
billings = pd.read_csv("../../data/01_raw/raw_billings.csv")
billings.shape
billings.info()
billings.head()

C:\Users\shrey\AppData\Local\Temp\ipykernel_32244\1029342165.py:1: DtypeWarning: Columns (15,16,19,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  billings = pd.read_csv("../../data/01_raw/raw_billings.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122082 entries, 0 to 122081
Data columns (total 59 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Co_Ref                      122082 non-null  object 
 1   Renewal_Month               122082 non-null  object 
 2   Connection_Net              8037 non-null    float64
 3   Connection_Qty              8037 non-null    float64
 4   Discount_Amount             13551 non-null   object 
 5   Sustainability_Score        122082 non-null  float64
 6   Total_Renewal_Score_New     122082 non-null  float64
 7   Starting_Connection_Net     8545 non-null    float64
 8   Starting_Connection_Qty     8545 non-null    float64
 9   Last_Years_Price            112992 non-null  float64
 10  Last_Years_Date_Paid        0 non-null       float64
 11  Auto_Renewal_Score          122082 non-null  int64  
 12  Status_Scores               122082 non-null  int64  
 13  Anchoring_Scor

,Co_Ref,Renewal_Month,Connection_Net,Connection_Qty,Discount_Amount,Sustainability_Score,Total_Renewal_Score_New,Starting_Connection_Net,Starting_Connection_Qty,Last_Years_Price,...,Connection_Group,Tenure_Group,#_of_Connection,Last_Renewal,Last_Band,Last_Total_Net_Paid,Last_Connections,Anchor_Group,Renewal_Year,DateTime_Out
0,VT6174,01-11-2024,NaN,NaN,NaN,8.0,42.5,NaN,NaN,799.0,...,1,3,1.0,01-11-2023,Band B,664.0,1.0,1,2024,01-11-2024
1,VD3828,01-08-2025,NaN,NaN,NaN,8.0,41.5,NaN,NaN,799.0,...,1,1,1.0,NaN,NaN,NaN,NaN,1,2025,01-08-2025
2,DV8120,01-03-2025,NaN,NaN,NaN,8.0,33.0,NaN,NaN,799.0,...,1,4+,1.0,01-03-2024,Band C1,749.0,1.0,1,2025,01-03-2025
3,EZ9894,01-06-2025,NaN,NaN,NaN,9.5,44.5,NaN,NaN,799.0,...,1,4+,1.0,01-06-2024,Band C1,749.0,1.0,1,2025,01-06-2025
4,FA8957,01-03-2025,NaN,NaN,NaN,9.5,42.5,NaN,NaN,799.0,...,1,3,1.0,01-03-2024,Band C1,749.0,1.0,1,2025,01-03-2025


In [53]:
billings = billings.drop_duplicates()
print("Shape after duplicate removal:", billings.shape)

Shape after duplicate removal: (122082, 59)


In [54]:
billings.columns = (
    billings.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [55]:
obj_cols = billings.select_dtypes(include="object").columns

for col in obj_cols:
    billings[col] = billings[col].astype(str).str.strip()

In [56]:
# check missing percentage
missing_pct = billings.isnull().mean() * 100
missing_summary = missing_pct.sort_values(ascending=False)
print("Missing values (%) by column:")
print(missing_summary.head(15))

last_years_date_paid        100.000000
connection_net               93.416720
connection_qty               93.416720
starting_connection_net      93.000606
starting_connection_qty      93.000606
last_connections             39.629102
last_total_net_paid          39.609443
payment_timeframe            17.083600
total_net_paid               17.083600
last_years_price              7.445815
tenure_years                  0.833866
#_of_connection               0.103209
proforma_approved_lists       0.103209
renewal_score_at_release      0.103209
starting_membership_net       0.000000
starting_package_net          0.000000
starting_pqq_net              0.000000
gross                         0.000000
membership_net                0.000000
package_net                   0.000000
dtype: float64

In [57]:
# columns with >70% missing values
high_missing_cols = missing_pct[missing_pct > 70].index

print(f"\nDropping {len(high_missing_cols)} columns with >70% missing:")
print(list(high_missing_cols))

billings = billings.drop(columns=high_missing_cols)

Dropping high-missing columns:
['connection_net', 'connection_qty', 'starting_connection_net', 'starting_connection_qty', 'last_years_date_paid']


In [58]:
date_cols = [
    "last_years_date_paid",
    "proforma_date",
    "registration_date",
    "prospect_renewal_date",
    "closed_date"
]

for col in date_cols:
    if col in billings.columns:
        billings[col] = pd.to_datetime(
            billings[col],
            errors="coerce"
        )

In [59]:
numeric_cols = [
    "discount_amount",
    "tenure_years",
    "amount",
    "total_amount",
    "gross",
    "membership_net",
    "package_net",
    "pqqnet",
    "total_net_paid",
    "starting_net",
    "starting_gross"
]

for col in numeric_cols:
    if col in billings.columns:
        billings[col] = pd.to_numeric(
            billings[col],
            errors="coerce"
        )

In [60]:
cat_cols = billings.select_dtypes(include="object").columns

for col in cat_cols:
    billings[col] = billings[col].replace("nan", np.nan)
    billings[col] = billings[col].fillna("Unknown")

In [61]:
num_cols = billings.select_dtypes(include=np.number).columns

for col in num_cols:
    billings[col] = billings[col].fillna(billings[col].median())

In [62]:
billings.isnull().sum().sort_values(ascending=False).head(20)

discount_amount            122082
closed_date                 76604
registration_date           75813
prospect_renewal_date       75136
proforma_date               16668
starting_gross                  0
starting_membership_net         0
starting_package_net            0
starting_pqq_net                0
gross                           0
membership_net                  0
package_net                     0
pqqnet                          0
total_net_paid                  0
prospect_outcome                0
amount                          0
payment_method                  0
starting_net                    0
total_amount                    0
connection_group                0
dtype: int64

In [ ]:
billings.to_csv("../../data/02_processed/processed_billings.csv", index=False)
print("✅ Processed billings saved!")

In [ ]:
print("Final shape:", billings.shape)
display(billings.head())

<bound method NDFrame.head of         co_ref renewal_month  discount_amount  sustainability_score  \
0       VT6174    01-11-2024              NaN                   8.0   
1       VD3828    01-08-2025              NaN                   8.0   
2       DV8120    01-03-2025              NaN                   8.0   
3       EZ9894    01-06-2025              NaN                   9.5   
4       FA8957    01-03-2025              NaN                   9.5   
...        ...           ...              ...                   ...   
122077  VI6211    01-04-2023              NaN                   8.0   
122078  NI7208    01-06-2023              NaN                   8.0   
122079  DO6023    01-09-2023              NaN                   8.0   
122080  KR5815    01-08-2023              NaN                   8.0   
122081  MA8737    01-07-2023              NaN                   8.0   

        total_renewal_score_new  last_years_price  auto_renewal_score  \
0                          42.5             

In [66]:
# 🔍 EXPLORATION: Remaining Data Quality Issues
print("="*60)
print("REMAINING DATA QUALITY CHECKS")
print("="*60)

# 1. Check for "Unknown" over-representation in categorical columns
print("\n1️⃣  CATEGORICAL COLUMNS - 'Unknown' Fill Rate:")
for col in billings.select_dtypes(include="object").columns:
    unknown_pct = (billings[col] == "Unknown").sum() / len(billings) * 100
    if unknown_pct > 5:
        print(f"   {col}: {unknown_pct:.1f}% Unknown")

# 2. Check for negative values in financial columns
print("\n2️⃣  NEGATIVE VALUES in Financial Columns:")
financial_cols = ["amount", "total_amount", "gross", "membership_net", "package_net", 
                  "total_net_paid", "starting_net", "starting_gross", "discount_amount"]
has_negatives = False
for col in financial_cols:
    if col in billings.columns:
        neg_count = (billings[col] < 0).sum()
        if neg_count > 0:
            print(f"   {col}: {neg_count} negative values")
            has_negatives = True
if not has_negatives:
    print("   ✅ None found")

# 3. Check date column issues
print("\n3️⃣  DATE COLUMNS - NaT (Missing) Percentage:")
date_cols_check = ["last_years_date_paid", "proforma_date", "registration_date", 
                   "prospect_renewal_date", "closed_date"]
for col in date_cols_check:
    if col in billings.columns:
        nat_pct = billings[col].isna().sum() / len(billings) * 100
        print(f"   {col}: {nat_pct:.1f}% NaT")

# 4. Check categorical value consistency (top values)
print("\n4️⃣  CATEGORICAL COLUMNS - Top Values (potential issues):")
for col in billings.select_dtypes(include="object").columns:
    top_val = billings[col].value_counts().head(1).index[0]
    top_pct = billings[col].value_counts().head(1).values[0] / len(billings) * 100
    if top_pct > 50:
        print(f"   {col}: '{top_val}' = {top_pct:.1f}% (⚠️ high concentration)")
    else:
        print(f"   {col}: {billings[col].nunique()} unique values")

# 5. Zero values in financial columns
print("\n5️⃣  ZERO VALUES in Financial Columns (potential data issues):")
for col in financial_cols:
    if col in billings.columns:
        zero_count = (billings[col] == 0).sum()
        zero_pct = zero_count / len(billings) * 100
        if zero_count > len(billings) * 0.1:  # More than 10%
            print(f"   {col}: {zero_count} zeros ({zero_pct:.1f}%)")

print("\n" + "="*60)

REMAINING DATA QUALITY CHECKS

1️⃣  CATEGORICAL COLUMNS - 'Unknown' Fill Rate:
   proforma_auto_renewal: 14.8% Unknown
   proforma_world_pay_token: 14.8% Unknown
   current_anchor_list: 21.3% Unknown
   proforma_account_stage: 7.6% Unknown
   proforma_audit_status: 7.6% Unknown
   last_renewal: 39.6% Unknown
   last_band: 39.6% Unknown

2️⃣  NEGATIVE VALUES in Financial Columns:
   ✅ None found

3️⃣  DATE COLUMNS - NaT (Missing) Percentage:
   proforma_date: 13.7% NaT
   registration_date: 62.1% NaT
   prospect_renewal_date: 61.5% NaT
   closed_date: 62.7% NaT

4️⃣  CATEGORICAL COLUMNS - Top Values (potential issues):
   co_ref: 47826 unique values
   renewal_month: 49 unique values
   proforma_auto_renewal: 'True' = 83.9% (⚠️ high concentration)
   proforma_world_pay_token: 'False' = 50.4% (⚠️ high concentration)
   current_anchor_list: 17056 unique values
   proforma_account_stage: 'Published' = 63.7% (⚠️ high concentration)
   proforma_audit_status: 'Accredited' = 62.8% (⚠️ high con

In [67]:

# 🧹 ADDITIONAL CLEANING OPERATIONS

print("\n📋 PERFORMING ADDITIONAL CLEANING...")

# 1. Drop date columns with >50% missing values
high_nat_dates = ["registration_date", "prospect_renewal_date", "closed_date"]
for col in high_nat_dates:
    if col in billings.columns:
        nat_pct = billings[col].isna().sum() / len(billings) * 100
        if nat_pct > 50:
            billings = billings.drop(columns=[col])
            print(f"   ✅ Dropped '{col}' ({nat_pct:.1f}% NaT)")

# 2. Drop low-variance categorical columns (>90% one value)
print("\n   Checking for low-variance categorical columns:")
low_var_cols = []
for col in billings.select_dtypes(include="object").columns:
    top_pct = billings[col].value_counts().head(1).values[0] / len(billings) * 100
    if top_pct > 90:
        low_var_cols.append(col)
        billings = billings.drop(columns=[col])
        print(f"   ✅ Dropped '{col}' ({top_pct:.1f}% one value)")

if not low_var_cols:
    print("   ✅ No extremely low-variance columns found")

# 3. Handle last_renewal and last_band with high Unknown rates
if "last_renewal" in billings.columns:
    unknown_pct = (billings["last_renewal"] == "Unknown").sum() / len(billings) * 100
    if unknown_pct > 30:
        billings["has_last_renewal"] = (billings["last_renewal"] != "Unknown").astype(int)
        print(f"   ✅ Created 'has_last_renewal' flag (Unknown: {unknown_pct:.1f}%)")

if "last_band" in billings.columns:
    unknown_pct = (billings["last_band"] == "Unknown").sum() / len(billings) * 100
    if unknown_pct > 30:
        billings["has_last_band"] = (billings["last_band"] != "Unknown").astype(int)
        print(f"   ✅ Created 'has_last_band' flag (Unknown: {unknown_pct:.1f}%)")

# 4. Convert yes/no binary columns to binary (0/1)
print("\n   Converting y/n columns to binary...")
yes_no_cols = [col for col in billings.columns 
               if billings[col].dtype == 'object' 
               and billings[col].nunique() <= 3
               and set(billings[col].unique()).issubset({'y', 'n', 'Unknown'})]

for col in yes_no_cols:
    billings[col] = billings[col].map({'y': 1, 'n': 0, 'Unknown': 0})
    print(f"   ✅ Converted '{col}' to binary")

# 5. Create binary indicators from True/False strings
print("\n   Converting True/False columns to binary...")
true_false_cols = [col for col in billings.columns 
                   if billings[col].dtype == 'object' 
                   and set(billings[col].unique()).issubset({'True', 'False', 'Unknown'})]

for col in true_false_cols:
    billings[col] = billings[col].map({'True': 1, 'False': 0, 'Unknown': 0}).astype('int64')
    print(f"   ✅ Converted '{col}' to numeric binary")

# 6. Verify no remaining nulls
print(f"\n   Final null check: {billings.isnull().sum().sum()} total nulls")

print("\n✅ Additional cleaning complete!")


📋 PERFORMING ADDITIONAL CLEANING...
   ✅ Dropped 'registration_date' (62.1% NaT)
   ✅ Dropped 'prospect_renewal_date' (61.5% NaT)
   ✅ Dropped 'closed_date' (62.7% NaT)

   Checking for low-variance categorical columns:
   ✅ Dropped 'current_auto_renewal_flag' (94.0% one value)
   ✅ Created 'has_last_renewal' flag (Unknown: 39.6%)
   ✅ Created 'has_last_band' flag (Unknown: 39.6%)

   Converting y/n columns to binary...
   ✅ Converted 'current_world_pay_token' to binary

   Converting True/False columns to binary...
   ✅ Converted 'proforma_auto_renewal' to numeric binary
   ✅ Converted 'proforma_world_pay_token' to numeric binary

   Final null check: 138750 total nulls

✅ Additional cleaning complete!


In [ ]:
# 🔄 FINAL NULL/UNKNOWN VALUE HANDLING

print("\n📋 FINAL NULL & UNKNOWN VALUE CLEANUP...")

# Re-run null-fill operations after dropping columns
print(f"   Before final cleanup: {billings.isnull().sum().sum()} nulls")

# Categorical columns: Replace "Unknown" with mode or a specific value
cat_cols_final = billings.select_dtypes(include="object").columns
for col in cat_cols_final:
    unknown_count = (billings[col] == "Unknown").sum()
    if unknown_count > 0:
        # Replace Unknown with the most common non-Unknown value
        mode_val = billings[billings[col] != "Unknown"][col].mode()
        if len(mode_val) > 0:
            billings[col] = billings[col].replace("Unknown", mode_val[0])
            print(f"   ✅ Replaced Unknown in '{col}' with '{mode_val[0]}'")

# Numeric columns: Fill remaining NaN with median
num_cols_final = billings.select_dtypes(include=np.number).columns
for col in num_cols_final:
    if billings[col].isnull().sum() > 0:
        billings[col] = billings[col].fillna(billings[col].median())

print(f"   After final cleanup: {billings.isnull().sum().sum()} nulls")
print("✅ Data is now clean and ready for modeling!")


📋 FINAL NULL & UNKNOWN VALUE CLEANUP...
   Before final cleanup: 138750 nulls
   ✅ Replaced Unknown in 'current_anchor_list' with 'CBRE'
   ✅ Replaced Unknown in 'proforma_account_stage' with 'Published'
   ✅ Replaced Unknown in 'proforma_audit_status' with 'Accredited'
   ✅ Replaced Unknown in 'proforma_membership_status' with 'Accredited'
   ✅ Replaced Unknown in 'band' with 'Band B'
   ✅ Replaced Unknown in 'connection_group' with '1'
   ✅ Replaced Unknown in 'tenure_group' with '4+'
   ✅ Replaced Unknown in 'last_renewal' with '01-03-2025'
   ✅ Replaced Unknown in 'last_band' with 'Band B'
   ✅ Replaced Unknown in 'anchor_group' with '1'
   After final cleanup: 138750 nulls
✅ Data is now clean and ready for modeling!


In [69]:

# Quick verification of remaining nulls
null_cols = billings.isnull().sum()
print("\n📊 REMAINING NULLS BY COLUMN:")
print(null_cols[null_cols > 0].sort_values(ascending=False).head(10))
print(f"\nTotal rows: {len(billings)}")
print(f"Total nulls: {null_cols.sum()}")


📊 REMAINING NULLS BY COLUMN:
discount_amount    122082
proforma_date       16668
dtype: int64

Total rows: 122082
Total nulls: 138750


In [70]:

# 🗑️ DROP COMPLETELY NULL COLUMNS & FINAL HANDLING

print("\n🗑️  FINAL DATA CLEANUP - Remove Useless Columns")

# Drop columns that are 100% null
for col in billings.columns:
    if billings[col].isnull().sum() == len(billings):
        billings = billings.drop(columns=[col])
        print(f"   ✅ Dropped '{col}' (100% null)")

# Handle proforma_date (has some NaT) - fill with a default date or drop
if "proforma_date" in billings.columns:
    nat_pct = billings["proforma_date"].isnull().sum() / len(billings) * 100
    # Fill NaT dates with median date
    billings["proforma_date"] = billings["proforma_date"].fillna(
        billings["proforma_date"].median()
    )
    print(f"   ✅ Filled '{nat_pct:.1f}% NaT in 'proforma_date' with median")

print(f"\n✅ FINAL DATA STATUS:")
print(f"   Shape: {billings.shape}")
print(f"   Total nulls: {billings.isnull().sum().sum()}")
print(f"   Memory usage: {billings.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Save the cleaned data
billings.to_csv("../../data/02_processed/processed_billings.csv", index=False)
print(f"   ✅ Saved to processed_billings.csv")


🗑️  FINAL DATA CLEANUP - Remove Useless Columns
   ✅ Dropped 'discount_amount' (100% null)
   ✅ Filled '13.7% NaT in 'proforma_date' with median

✅ FINAL DATA STATUS:
   Shape: (122082, 51)
   Total nulls: 0
   Memory usage: 143.55 MB
   ✅ Saved to processed_billings.csv
